# Agenti 

![agenti](agenti.png)

In [ ]:
import os
import pandas as pd
from sklearn.decomposition import PCA
import seaborn as sns
import matplotlib.pyplot as plt

from smolagents import CodeAgent, DuckDuckGoSearchTool, tool, InferenceClientModel

 # Různí agenti, různé výsledky

In [ ]:
model = InferenceClientModel(
    max_tokens=500,
    temperature=0.5,
    model_id="google/gemma-3-27b-it",
    token=os.getenv("HF_API_TOKEN")
)

In [ ]:
agent = CodeAgent(tools=[DuckDuckGoSearchTool()], model=model)
agent.run("How long it would take to an elephant to walk from Prague to Liberec?")

### Úkol: Jak by dopadl stejný dotaz pro jiný model? Vytvořte si model_2 ve kterém použijete LLM: "meta-llama/Llama-3.3-70B-Instruct"

Co jsou to ty tokeny? https://huggingface.co/learn/agents-course/unit1/what-are-llms

![proces](proces.png)

# Jak si vytvořit nástroj

In [ ]:
pd.read_csv('penguins_size_nona.csv').head()

In [ ]:
@tool
def pca_csv(input_path: str, n_components: int = 2) -> pd.DataFrame:
    """
    Apply PCA to a CSV file and reduce to n_components (default 2).

    Args:
        input_path (str): Path to the CSV file.
        n_components (int): Number of principal components.

    Returns:
        dict: DataFrame with the principal components
    """
    # Load CSV
    df = pd.read_csv(input_path)

    # Keep only numeric columns
    numeric_df = df.select_dtypes(include='number')
    if numeric_df.empty:
        raise ValueError("CSV has no numeric columns for PCA")

    # Apply PCA
    pca = PCA(n_components=n_components)
    components = pca.fit_transform(numeric_df)

    # Create DataFrame for components
    pc_df = pd.DataFrame(components, columns=[f"PC{i+1}" for i in range(n_components)])
    return pc_df


In [ ]:
@tool
def plot_scatter_auto(df: pd.DataFrame, title: str = None) -> str:
    """
    Creates a Seaborn scatterplot from a DataFrame.
    Automatically picks the first two numeric columns.

    Args:
        df (pd.DataFrame): DataFrame containing the data.
        title (str, optional): Title of the plot.
    Returns:
        str: saying what are x and y columns
    """
    # Select numeric columns
    numeric_cols = df.select_dtypes(include='number').columns.tolist()
    
    if len(numeric_cols) < 2:
        raise ValueError("DataFrame must have at least two numeric columns for a scatterplot.")
    
    x, y = numeric_cols[:2]  # automatically pick first two numeric columns
    
    plt.figure(figsize=(8, 6))
    sns.scatterplot(data=df, x=x, y=y)
    
    if title:
        plt.title(title)
    
    plt.xlabel(x)
    plt.ylabel(y)
    plt.grid(True)
    plt.show()

    return f"Scatterplot created using columns '{x}' and '{y}'."

In [ ]:
agent = CodeAgent(tools=[pca_csv, plot_scatter_auto], additional_authorized_imports=['pandas'] , model=model)
agent.run("For data in penguins_size_nona.csv find the main two components and plot them into graph.")

### Úkol: Vytvořte vlastní nástroj (s dekorátorem tool), který vykreslí histogram hmotnosti tučňáků (sloupec obsahující "body_mass").

In [ ]:
from smolagents import Tool
from huggingface_hub import InferenceClient

class TextToImageTool(Tool):
    description = "This tool creates an image according to a prompt, which is a text description."
    name = "image_generator"
    inputs = {"prompt": {"type": "string", "description": "The image generator prompt."}}
    output_type = "image"
    model_sdxl = "black-forest-labs/FLUX.1-schnell"
    client = InferenceClient(model_sdxl)


    def forward(self, prompt):
        image = self.client.text_to_image(prompt)
        image.save('image.png')

        return f"Successfully saved image with this prompt {prompt}"

In [ ]:
image_generator = TextToImageTool()

agent = CodeAgent(tools=[image_generator] , model=model)
agent.run("Improve this prompt, then generate an image of it. Prompt: A cat swimming in the small pond.")

# Ale mnoho nástrojů již existuje

In [ ]:
agent = CodeAgent(
    tools=[],
    model=model,
    additional_authorized_imports=["duckdb"]
)

In [ ]:
prompt = """
You are a data analyst.
Your job is to answer questions asked by the user
about the given dataset, Income_Urban_VS_Rural.csv

The file Income_Urban_VS_Rural.csv is located in the current working directory.

## Data dictionary

1. County: County name.
2. State: State name.
3. FIPS: Combined state and county FIPS code.
4. State FIPS Code: State's Federal Information Processing Standard code.
5. County FIPS Code: County's FIPS code.
6. Total Population: Total population of the county.
7. Household Income: Median household income for the county.
8. Urban-Rural: Classification based on population (Urban or Rural).

* You can use duckdb to query the data.
"""

In [ ]:

question = """What is the average population size of urban counties
compared to rural counties?"""

agent.run(
    prompt + "\n" + question,
    additional_args = dict(source_file="Income_Urban_VS_Rural.csv")
)

### Úkol: Zjistěte, které okresy (conties) jsou nejvýznamnější z hlediska příjmu domácností.

# Více agentů

In [ ]:
import requests
from requests.exceptions import RequestException
from markdownify import markdownify

@tool
def visit_webpage(url: str) -> str:
    """Visits a webpage at the given URL and returns its content as a markdown string.

    Args:
        url: The URL of the webpage to visit.

    Returns:
        The content of the webpage converted to Markdown, or an error message if the request fails.
    """
    try:
        # Send a GET request to the URL
        response = requests.get(url)
        response.raise_for_status()  # Raise an exception for bad status codes

        # Convert the HTML content to Markdown
        markdown_content = markdownify(response.text).strip()

        # Remove multiple line breaks
        markdown_content = re.sub(r"\n{3,}", "\n\n", markdown_content)

        return markdown_content

    except RequestException as e:
        return f"Error fetching the webpage: {str(e)}"
    except Exception as e:
        return f"An unexpected error occurred: {str(e)}"

In [ ]:
from smolagents import VisitWebpageTool

web_agent = CodeAgent(
    model=model,
    tools=[
        DuckDuckGoSearchTool(),
        VisitWebpageTool(),
    ],
    name="web_agent",
    description="Browses the web to find information",
    verbosity_level=0,
    max_steps=10,
)

manager_agent = CodeAgent(
    model=model,
    tools=[],
    managed_agents=[web_agent],
    planning_interval=5,
    verbosity_level=2,
    max_steps=15,
)

manager_agent.run("Who is the elected president of United States in 2024?")